# 03 — Splitting Text Into Smart Chunks (Sizes, Overlaps & Trade-offs)

**By the end of this notebook, you can split long files with `RecursiveCharacterTextSplitter`, avoid cutting answers in half, and benchmark 100 vs 300 vs 800 chunk sizes.**

### Learning Objectives
- Visualize how `chunk_overlap` builds a semantic bridge across split boundaries.
- Demonstrate a real bad split cutting an answer in half and watch overlap repair it.
- Run a concrete experiment comparing chunk sizes 100, 300, and 800 on the same query while preserving metadata.

### Prerequisites
- Notebooks 00–02

## 1. Setup
Load `.env` configuration via `course/utils/models.py` and initialize our shared LLM and embeddings helpers.

In [1]:
# If running in Google Colab, uncomment: !git clone https://github.com/rahat4528/rag-from-scratch-course-v2.git && %cd rag-from-scratch-course-v2 && pip install -q -r course/requirements.txt
import sys
from pathlib import Path
repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'course' / 'utils').exists()), Path.cwd())
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
from course.utils.models import get_llm, get_embeddings, resolve_course_root
course_dir = resolve_course_root(); data_dir = course_dir / 'data'
llm = get_llm(fallback_to_demo=True); embeddings = get_embeddings()
print(f'Ready | LLM={type(llm).__name__} | Embeddings={type(embeddings.base).__name__}')

[OfflineDemoChatModel] Notice: Provider 'ollama' not active (Ollama server is not reachable at http://localhost:11434. Install/start Ollama (`ollama pull llama3.2`) or switch RAG_PROVIDER in .env.). Running code-path demo mode.


W1112 00:36:52.153000 2372 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready | LLM=OfflineDemoChatModel | Embeddings=LocalSentenceTransformerEmbeddings


## 2. Concept: The Cliffhanger Bridge: Why Overlap Exists

Imagine cutting a recipe right between *"Preheat oven to"* and *"220 degrees Celsius"*. If search pulls only the second half, crucial context vanishes!

- **Chunking** breaks multi-page files into focused chunks so vector search targets specific facts cleanly.
- **Chunk Overlap** repeats the last 15–20% of chunk *N* at the top of chunk *N+1* like a cliffhanger recap across chapters.
- **RecursiveCharacterTextSplitter** prefers natural breaks (`\n\n` -> `\n` -> ` ` -> char) instead of slicing mid-word.

```
Source: [ Verify port 9872. Ensure clock drift < 5 ms. Restart novasphere-edge daemon. ]

No overlap  (bad): [ ...Ensure clock ] | [ drift < 5 ms... ]  <-- Split right across "< 5 ms"!
With overlap(fix): [ ...Ensure clock drift < 5 ms. ]
                              [ drift < 5 ms. Restart novasphere-edge daemon. ]
```

## 3. Step-by-Step Build

### Step 1: Demonstrate a Bad Split vs. Overlap Fix
Look at what happens when `chunk_size=100, chunk_overlap=0` chops right through `ERR_QUANTUM_TIMEOUT` resolution steps vs adding `chunk_overlap=35`.

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
faq_text = (data_dir / "product_faq.md").read_text(encoding="utf-8")
target = [c for c in RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0).split_text(faq_text) if "9872" in c][0]
fixed_chunks = RecursiveCharacterTextSplitter(chunk_size=180, chunk_overlap=50).split_text(faq_text)
target_fix = [c for c in fixed_chunks if "9872" in c][0]
print("Bad no-overlap chunk lacks '< 5 ms'?:", "< 5 ms" not in target, "| snippet:", repr(target[:75]))
print("Fixed overlap chunk keeps context?  :", "QuantumCache" in target_fix or "port 9872" in target_fix)
        

Bad no-overlap chunk lacks '< 5 ms'?: True | snippet: '9872**.'
Fixed overlap chunk keeps context?  : True


> **What just happened?**
> Zero-overlap tiny splits stranded `port 9872` away from `ERR_QUANTUM_TIMEOUT` and `< 5 ms`!

### Step 2: Preserve Metadata Automatically Across Splits
Passing `Document` objects into `.split_documents()` copies `.metadata` (`source`, `access_level`, `doc_type`) onto EVERY child chunk.

In [3]:
from course.utils.helpers import load_corpus_documents
docs = load_corpus_documents(data_dir)
chunks = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50).split_documents(docs)
print(f"Expanded {len(docs)} files -> {len(chunks)} chunks. First chunk meta:", chunks[0].metadata)
assert all("source" in c.metadata for c in chunks)
        

Expanded 4 files -> 38 chunks. First chunk meta: {'source': 'company_handbook.txt', 'doc_type': 'handbook', 'access_level': 'employee', 'doc_id': 'doc-handbook'}


> **What just happened?**
> Every chunk remembers its source filename and RBAC access level automatically.

### Step 3: Experiment Table — Chunk Sizes 100 vs. 300 vs. 800 on One Question
Let's index our corpus at chunk sizes `100`, `300`, and `800` (with 15% overlap), search for `ERR_QUANTUM_TIMEOUT` steps (`port 9872`, `< 5 ms`, `systemctl`), and tabulate complete step coverage!

In [4]:
import pandas as pd
from langchain_core.vectorstores import InMemoryVectorStore
q = "What steps resolve ERR_QUANTUM_TIMEOUT?"
rows = []
for sz in (100, 300, 800):
    ch = RecursiveCharacterTextSplitter(chunk_size=sz, chunk_overlap=int(sz*0.15)).split_documents(docs)
    top1 = InMemoryVectorStore.from_documents(ch, embeddings).similarity_search(q, k=1)[0].page_content
    has_all = all(tok in top1 for tok in ("9872", "< 5 ms", "novasphere-edge"))
    rows.append({"ChunkSize": sz, "TotalChunks": len(ch), "Top1Len": len(top1), "HasAll3Steps": has_all})
print(pd.DataFrame(rows).to_string(index=False))
        

 ChunkSize  TotalChunks  Top1Len  HasAll3Steps
       100          107       54         False
       300           38      233         False
       800           13      621          True


> **What just happened?**
> Too small (`100`) splits the steps apart; `800` captures all 3 steps inside Chunk #1 (though very large chunks can dilute precision on dense docs).

## 4. Try It Yourself
1. **Test invalid overlap > chunk_size** — *Hint:* Observe how LangChain protects against `chunk_overlap > chunk_size`.
2. **Verify zero lost text on tiny document** — *Hint:* Check `.split_text('Hi RAG!')` preserves exact text.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>
Run the cell below to check one working solution.
</details>

In [5]:
try:
    RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=80)
except ValueError as err:
    print("Caught overlap > size cleanly:", str(err)[:85])
print("Tiny doc:", RecursiveCharacterTextSplitter(chunk_size=300).split_text("Hi RAG!"))
        

Caught overlap > size cleanly: Got a larger chunk overlap (80) than chunk size (50), should be smaller.
Tiny doc: ['Hi RAG!']


## 5. Common Mistakes
- Picking `chunk_size=100` blindly — shatters numbered instructions and tables.
- Setting `chunk_overlap=0` — orphans boundary words right between chunks.
- Forgetting `.split_documents()` and calling `.split_text()`, losing all metadata.

## 6. Recap
- `RecursiveCharacterTextSplitter` breaks at paragraphs, lines, and spaces cleanly.
- Overlap (~15%) bridges boundaries without throwing away context.
- Chunk sizes around 300–650 chars balance pinpoint vector precision against intact answers.

## 7. Next Notebook
Next (`04_embeddings_and_vector_stores.ipynb`): Turn chunks into geometric vectors using NumPy math, plot them in 2D, and persist a local vector store!

## 8. Video Chapter Outline
1. **Why We Split Documents** — Balancing focus vs completeness.
2. **Bad Split vs Overlap Fix Demo** — Rescuing cut instructions cleanly.
3. **Metadata Inheritance** — Tracing chunks back to source files.
4. **Chunk Size Experiment (100 / 300 / 800)** — Live Pandas measurement table.
5. **Recap & Safeguards** — Guarding overlap errors and tiny files.